In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PASTA_RAW   = Path("../data/raw")
PASTA_PROCESSED = Path("../data/processed")

#carregamento de dados climáticos dos 20 municípios
clima = pd.read_csv(PASTA_RAW / "clima_municipios_mg.csv",
                    parse_dates=["data"])
#carrega a produtividade histórica
safras = pd.read_csv(PASTA_RAW / "produtividade_mg.csv")

print("=== CLIMA ===")
print(f"LInhas totais: {len(clima)}")
print(f"Municípios: {clima['municipio'].nunique()}")
print(f"Regiões: {clima['regiao'].nunique()}")
print(f"Período: {clima['data'].min()} até {clima['data'].max()}")
print(clima.groupby(["regiao", "municipio"]).size().reset_index(name="dias"))

print("\n=== SAFRAS ===")
print(f"Linhas : {len(safras)}")



=== CLIMA ===
LInhas totais: 182640
Municípios: 20
Regiões: 6
Período: 2000-01-01 00:00:00 até 2024-12-31 00:00:00
            regiao            municipio  dias
0   Alto Paranaiba      carmo_paranaiba  9132
1   Alto Paranaiba       patos_de_minas  9132
2   Alto Paranaiba  presidente_olegario  9132
3   Alto Paranaiba        rio_paranaiba  9132
4          Central        para_de_minas  9132
5          Central               pompeu  9132
6          Central          sete_lagoas  9132
7         Noroeste     brasilania_minas  9132
8         Noroeste        joao_pinheiro  9132
9         Noroeste             paracatu  9132
10    Sul de Minas              alfenas  9132
11    Sul de Minas               lavras  9132
12    Sul de Minas               passos  9132
13    Sul de Minas      pocos_de_caldas  9132
14       Triangulo                araxa  9132
15       Triangulo            ituiutaba  9132
16       Triangulo              uberaba  9132
17       Triangulo           uberlandia  9132
18    Zona 

In [3]:
#colunas auxiliares

clima["ano"] = clima["data"].dt.year
clima["mes"] = clima["data"].dt.month

#filtrar somente meses do cilco do milho (out-mar)
clima_ciclo = clima[clima["mes"].isin([10, 11, 12, 1, 2, 3])].copy()

#definiçao da safra
def definir_safra(row):
    if row["mes"] >= 10:
        return row["ano"] + 1
    else:
        return row["ano"]
    
clima_ciclo["safra"] = clima_ciclo.apply(definir_safra, axis=1)

#Filtragem completa da safra
clima_ciclo = clima_ciclo[
    (clima_ciclo["safra"] >= 2001) & (clima_ciclo["safra"] <= 2024)
].copy()

print(f"Dias no ciclo do milho: {len(clima_ciclo)}")
print(f"Safras identificadas : {clima_ciclo['safra'].nunique()}")
print(f"Municípios: {clima_ciclo['municipio'].nunique()}")


Dias no ciclo do milho: 87480
Safras identificadas : 24
Municípios: 20


In [7]:
#Agrega safra e município

resumo = clima_ciclo.groupby(["safra", "municipio", "regiao"]).agg(
    chuva_total =   ("chuva_mm", "sum"),
    temp_media =    ("temp_max", "mean"),
    temp_min_media = ("temp_min", "mean"),
    et0_total = ("evapotranspiracao", "sum"),
    dias_sem_chuva = ("chuva_mm", lambda x: (x == 0).sum())
).reset_index()

#Acionamento da variável tendência tecnológica
resumo["tendencia"] = resumo["safra"] - resumo["safra"].min()

#união com produtividade
df = resumo.merge(safras, on="safra")

print(f"Linhas totais: {len(df)}")
print(f"Safras: {df['safra'].nunique()}")
print(f"Municípios: {df['municipio'].nunique()}")
print(f"regiões: {df['regiao'].nunique()}")
print(f"\nLinhas por Região: ")
print(df.groupby("regiao").size().reset_index(name="linhas"))
print(f"\nPrimeiras linhas: ")
print(df.head(10).round(2))


Linhas totais: 480
Safras: 24
Municípios: 20
regiões: 6

Linhas por Região: 
           regiao  linhas
0  Alto Paranaiba      96
1         Central      72
2        Noroeste      72
3    Sul de Minas      96
4       Triangulo      96
5    Zona da Mata      48

Primeiras linhas: 
   safra         municipio          regiao  chuva_total  temp_media  \
0   2001           alfenas    Sul de Minas        875.5       27.14   
1   2001             araxa       Triangulo        715.1       26.72   
2   2001  brasilania_minas        Noroeste        939.2       29.99   
3   2001   carmo_paranaiba  Alto Paranaiba        801.4       26.33   
4   2001         ituiutaba       Triangulo        930.7       29.52   
5   2001     joao_pinheiro        Noroeste        800.4       28.45   
6   2001            lavras    Sul de Minas        779.2       26.63   
7   2001            muriae    Zona da Mata        821.2       29.78   
8   2001     para_de_minas         Central        705.0       27.46   
9   2001   

In [8]:
#salva o dataset expandido

PASTA_PROCESSED.mkdir(exist_ok = True)
df.to_csv(PASTA_PROCESSED / "safras_municipios_mg.csv", index = False)
print("Arquivo salvo em data/processed/safras_municipios_mg.csv")

#correlações por regiao
print("\n=== CORRELAÇÃO CLIMA x PRODUTIVIDADE POR REGIÃO ===")

features = ["chuva_total", "temp_media", "et0_total", "dias_sem_chuva", "tendencia"]

for regiao in sorted(df["regiao"].unique()):
    df_reg = df[df['regiao'] == regiao]
    print(f"---{regiao} ({len(df_reg)} linhas) ---")
    for col in features:
        corr = df_reg[col].corr(df_reg["produtividade_sc_ha"])
        sinal = "+" if corr > 0 else "-"
        barra = "🟩" * int(abs(corr) * 10)
        print(f" {col:20s}: {corr:+.3f} {barra}")
        print()

Arquivo salvo em data/processed/safras_municipios_mg.csv

=== CORRELAÇÃO CLIMA x PRODUTIVIDADE POR REGIÃO ===
---Alto Paranaiba (96 linhas) ---
 chuva_total         : -0.234 🟩🟩

 temp_media          : +0.622 🟩🟩🟩🟩🟩🟩

 et0_total           : +0.594 🟩🟩🟩🟩🟩

 dias_sem_chuva      : -0.233 🟩🟩

 tendencia           : +0.916 🟩🟩🟩🟩🟩🟩🟩🟩🟩

---Central (72 linhas) ---
 chuva_total         : -0.174 🟩

 temp_media          : +0.370 🟩🟩🟩

 et0_total           : +0.405 🟩🟩🟩🟩

 dias_sem_chuva      : -0.307 🟩🟩🟩

 tendencia           : +0.916 🟩🟩🟩🟩🟩🟩🟩🟩🟩

---Noroeste (72 linhas) ---
 chuva_total         : -0.174 🟩

 temp_media          : +0.493 🟩🟩🟩🟩

 et0_total           : +0.490 🟩🟩🟩🟩

 dias_sem_chuva      : -0.007 

 tendencia           : +0.916 🟩🟩🟩🟩🟩🟩🟩🟩🟩

---Sul de Minas (96 linhas) ---
 chuva_total         : -0.305 🟩🟩🟩

 temp_media          : +0.266 🟩🟩

 et0_total           : +0.399 🟩🟩🟩

 dias_sem_chuva      : -0.272 🟩🟩

 tendencia           : +0.916 🟩🟩🟩🟩🟩🟩🟩🟩🟩

---Triangulo (96 linhas) ---
 chuva_total       

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import mean_absolute_error, r2_score
import pickle
import json

#features e targets

X = df[["chuva_total", "temp_media", "temp_min_media", 
        "et0_total", "dias_sem_chuva", "tendencia"]]
y = df["produtividade_sc_ha"]

#normalização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

#cross-validation com 5 rounds
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

#RandomForest com parâmetros ajustados para 480 linhas modeladas
modelo_rf = RandomForestRegressor(
    n_estimators = 200, #mais árvores
    max_depth = 7, #mais profundo que na v2
    min_samples_split = 4,
    min_samples_leaf = 42
)

scores = cross_val_score(modelo_rf, X_scaled, y, cv=kfold, scoring="r2")

print("=== RANDOM FOREST - MÉTRICAS ===\n")
print(f"R² por rodada: {scores.round(3)}")
print(f"R² médio: {scores.mean():.3f}")
print(f"Desvio Padrão: {scores.std():.3f}")


=== RANDOM FOREST - MÉTRICAS ===

R² por rodada: [0.882 0.887 0.862 0.877 0.901]
R² médio: 0.882
Desvio Padrão: 0.013


In [11]:
#treinamento do modelo e salvamento

#treinamento com todos os dados:
modelo_rf.fit(X_scaled, y)

#previsoes com cross-validation:
y_pred = cross_val_predict(modelo_rf, X_scaled, y, cv=kfold)

#metricas finais
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("=== MÉTRICAS FINAIS ===\n")
print(f"R²: {r2:.3f}")
print(f"MAE (MEAN ABSOLUTE ERROR): {mae:.3f} sc/ha")

#salva o modelo e o scaler

with open("../outputs/modelo_v3.pkl", "wb") as f:
    pickle.dump(modelo_rf, f)

with open("../outputs/scaler_v3.pkl", "wb") as f:
    pickle.dump(scaler, f)

#salva as métricas

metricas = {
    "modelo"        : "RandomForest v2.0",
    "r2"            : round(r2, 3),
    "n_linhas"      : round(mae, 2),
    "n_municipios"  : len(df),
    "n_splits"      : 5
}

with open("../outputs/metricas_v3.json", "w") as f:
    json.dump(metricas, f, indent=2)

print("\nArquivos salvos:")
print("     outputs/modelo_v3_pkl")
print("     outputs/scaler_v3.pkl")
print("     outputs/metricas_v3.json")


=== MÉTRICAS FINAIS ===

R²: 0.884
MAE (MEAN ABSOLUTE ERROR): 3.730 sc/ha

Arquivos salvos:
     outputs/modelo_v3_pkl
     outputs/scaler_v3.pkl
     outputs/metricas_v3.json


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

#previsões ao dataset
df["previsto"] = y_pred.round(1)
df["erro"]     = (df["previsto"] - df["produtividade_sc_ha"]).round(1)
margem         = df["erro"].std()

# calcula média por safra para linha principal
media_safra = df.groupby("safra").agg(
    real     = ("produtividade_sc_ha", "mean"),
    previsto = ("previsto",            "mean")
).reset_index()

# cores por região
cores_regiao = {
    "Triangulo"     : "#1D9E75",
    "Alto Paranaiba": "#BA7517",
    "Sul de Minas"  : "#378ADD",
    "Central"       : "#D85A30",
    "Noroeste"      : "#7F77DD",
    "Zona da Mata"  : "#D4537E"
}

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

#grafico 1: evolução geral 
ax1 = fig.add_subplot(gs[0, :])

ax1.fill_between(media_safra["safra"],
                 media_safra["previsto"] - margem,
                 media_safra["previsto"] + margem,
                 color="#BA7517", alpha=0.15,
                 label=f"Margem ±{margem:.1f} sc/ha")

ax1.plot(media_safra["safra"], media_safra["real"],
         "o-", color="#1D9E75", linewidth=2.5,
         markersize=7, label="Real — média MG (CONAB)", zorder=3)

ax1.plot(media_safra["safra"], media_safra["previsto"],
         "s--", color="#BA7517", linewidth=2,
         markersize=6, label=f"AgroPredict v2.0 — R² 0.884")

texto = (
    f"v1.0 → R² 0.716 · MAE 5.18 sc/ha\n"
    f"v1.2 → R² 0.796 · MAE 4.78 sc/ha\n"
    f"v2.0 → R² 0.884 · MAE 3.70 sc/ha"
)
ax1.text(0.02, 0.97, texto,
         transform=ax1.transAxes, fontsize=9,
         verticalalignment="top",
         bbox=dict(boxstyle="round", facecolor="white",
                   edgecolor="#BA7517", alpha=0.8))

ax1.set_title("AgroPredict v2.0 — Evolução da previsão\n"
              "Milho 1ª safra · 20 municípios · Minas Gerais · 2001–2024",
              fontsize=13, fontweight="bold")
ax1.set_xlabel("Safra")
ax1.set_ylabel("Produtividade média (sc/ha)")
ax1.set_xticks(media_safra["safra"])
ax1.set_xticklabels(media_safra["safra"], rotation=45, ha="right")
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

#graficos de 2 a 7: um por região 
regioes = sorted(df["regiao"].unique())
posicoes = [(1,0),(1,1),(2,0),(2,1),(2,0),(2,1)]
posicoes = [gs[1,0], gs[1,1], gs[2,0], gs[2,1]]

for i, regiao in enumerate(regioes[:4]):
    ax = fig.add_subplot(posicoes[i])
    df_reg = df[df["regiao"] == regiao]

    media_reg = df_reg.groupby("safra").agg(
        real     = ("produtividade_sc_ha", "mean"),
        previsto = ("previsto",            "mean")
    ).reset_index()

    cor = cores_regiao[regiao]
    ax.plot(media_reg["safra"], media_reg["real"],
            "o-", color=cor, linewidth=2, markersize=5,
            label="Real")
    ax.plot(media_reg["safra"], media_reg["previsto"],
            "s--", color=cor, linewidth=1.5, markersize=4,
            alpha=0.7, label="Previsto")

    r2_reg  = r2_score(df_reg["produtividade_sc_ha"], df_reg["previsto"])
    mae_reg = mean_absolute_error(df_reg["produtividade_sc_ha"], df_reg["previsto"])

    ax.set_title(f"{regiao}\nR² {r2_reg:.3f} · MAE {mae_reg:.1f} sc/ha",
                 fontsize=10)
    ax.set_xlabel("Safra", fontsize=8)
    ax.set_ylabel("sc/ha", fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("AgroPredict v2.0 — Desempenho por região",
             fontsize=14, fontweight="bold", y=1.01)

plt.savefig("../outputs/agropredict_v3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo em outputs/agropredict_v3.png")